In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('df_filtered.csv')


# Filter the dataset to keep only "Parkinson's" and "Other_Disorders", discarding "Healthy"
df_filtered = df[df['Rencoded'].isin(["Parkinson's", 'Other_Disorders'])]

# Display the first few rows of the dataset to understand its structure
df.head()

,resource_type_x,id_x,study_id_x,condition,disease_comment,age_at_diagnosis,age,height,weight,gender,...,Daytime sleepiness,Insomnia,Intense vivid dreams,Acting out during dreams,Restless legs,Rencoded,condition_Healthy,condition_Other_Disorders,condition_Parkinson's,Rencoded_numeric
0,patient,2,PADS,Other Movement Disorders,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,...,1,1,0,1,0,Other_Disorders,False,True,False,2
1,patient,4,PADS,Parkinson's,IPS akinetic-rigid type,63,67,161,90,female,...,1,1,0,0,1,Parkinson's,False,False,True,1
2,patient,5,PADS,Parkinson's,IPS tremordominant type,65,75,172,86,male,...,1,1,1,1,1,Parkinson's,False,False,True,1
3,patient,6,PADS,Parkinson's,IPS currently inpatient treatment for epilepsy...,60,72,171,115,female,...,0,1,0,0,1,Parkinson's,False,False,True,1
4,patient,7,PADS,Other Movement Disorders,Atypical IPS,73,74,181,94,male,...,0,1,0,0,1,Other_Disorders,False,True,False,2


In [2]:
df_filtered.shape

(390, 55)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix, roc_curve, auc)
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from collections import Counter
from imblearn.over_sampling import SMOTE
symptom_columns = [
    'Dribbling', 'Swallowing', 'Vomiting', 'Constipation', 'Bowel inconsistence',
    'Bowel emptying incomplete', 'Urgency', 'Nocturia', 'Pains', 'Weight', 'Sweating',
    'Diplopia', 'Remembering', 'Loss of interest', 'Concentrating', 'Taste/smelling',
    'Delusions', 'Sad, blues', 'Anxiety', 'Sex difficulty',
    'Falling', 'Swelling', 'Daytime sleepiness', 'Insomnia', 'Intense vivid dreams',
    'Acting out during dreams', 'Restless legs'
]


X = df[symptom_columns]
y = df_filtered['Rencoded'].apply(lambda x: 1 if x == "Parkinson's" else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**PCA**

In [11]:
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier()
}

# Apply PCA
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with PCA...")

    fold_results = []  # To collect results for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_pca, y_train)):
        X_fold_train, X_fold_val = X_train_pca[train_idx], X_train_pca[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the entire training set for testing
    model.fit(X_train_pca, y_train)

    # Get predictions for testing set
    y_test_pred = model.predict(X_test_pca)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions in the specified format
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set in the specified format
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    weighted_test_precision = test_report_dict['weighted avg']['precision']
    weighted_test_recall = test_report_dict['weighted avg']['recall']
    weighted_test_f1 = test_report_dict['weighted avg']['f1-score']

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")


Training KNN with PCA...

Weighted Average (Training) for KNN:
Precision: 0.77, Recall: 0.84, F1-Score: 0.81

Accuracy for KNN (Testing): 0.71

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.48      0.49        23
   Parkinson's       0.79      0.80      0.79        55

      accuracy                           0.71        78
     macro avg       0.64      0.64      0.64        78
  weighted avg       0.70      0.71      0.70        78


Confusion Matrix for KNN (Testing):
[[11 12]
 [11 44]]

Weighted Average (Testing) for KNN:
Precision: 0.70, Recall: 0.71, F1-Score: 0.70

Training Naive Bayes with PCA...

Weighted Average (Training) for Naive Bayes:
Precision: 0.74, Recall: 0.88, F1-Score: 0.80

Accuracy for Naive Bayes (Testing): 0.73

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.58      0.30      0.40        23
   Pa

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1


Weighted Average (Training) for AdaBoost:
Precision: 0.76, Recall: 0.85, F1-Score: 0.80

Accuracy for AdaBoost (Testing): 0.71

Classification Report for AdaBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.43      0.47        23
   Parkinson's       0.78      0.82      0.80        55

      accuracy                           0.71        78
     macro avg       0.64      0.63      0.63        78
  weighted avg       0.69      0.71      0.70        78


Confusion Matrix for AdaBoost (Testing):
[[10 13]
 [10 45]]

Weighted Average (Testing) for AdaBoost:
Precision: 0.69, Recall: 0.71, F1-Score: 0.70

Training XGB with PCA...


/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



Weighted Average (Training) for XGB:
Precision: 0.77, Recall: 0.85, F1-Score: 0.80

Accuracy for XGB (Testing): 0.74

Classification Report for XGB (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.59      0.43      0.50        23
   Parkinson's       0.79      0.87      0.83        55

      accuracy                           0.74        78
     macro avg       0.69      0.65      0.66        78
  weighted avg       0.73      0.74      0.73        78


Confusion Matrix for XGB (Testing):
[[10 13]
 [ 7 48]]

Weighted Average (Testing) for XGB:
Precision: 0.73, Recall: 0.74, F1-Score: 0.73

Training Extra Trees with PCA...

Weighted Average (Training) for Extra Trees:
Precision: 0.76, Recall: 0.90, F1-Score: 0.82

Accuracy for Extra Trees (Testing): 0.76

Classification Report for Extra Trees (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.60      0.52      0.56        23
   Parkinson's       0.81      

In [8]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 9.3 MB/s eta 0:00:00


SVM, MLP, cat n polynomial

In [13]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

# Define models
models = {
    'SVM': SVC(kernel='rbf', C=1, gamma='scale', probability=True, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, activation='relu', solver='adam', random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': make_pipeline(PolynomialFeatures(degree=2), LogisticRegression(max_iter=1000)),
}

# Apply PCA to reduce dimensions
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Stratified K-Fold cross-validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with PCA...")

    fold_results = []  # Collect results for each fold

    # Cross-validation loop
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_pca, y_train)):
        # Select appropriate dataset
        if name == 'CatBoost':  # CatBoost does not require PCA
            X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            X_train_final, X_test_final = X_train_scaled, X_test_scaled
        else:
            X_fold_train, X_fold_val = X_train_pca[train_idx], X_train_pca[val_idx]
            X_train_final, X_test_final = X_train_pca, X_test_pca

        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train model
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Store fold results
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the full training dataset for testing
    model.fit(X_train_final, y_train)
    y_test_pred = model.predict(X_test_final)

    # Compute testing accuracy
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print classification report for the test set
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Print confusion matrix for the test set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Weighted averages for the test results
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    weighted_test_precision = test_report_dict['weighted avg']['precision']
    weighted_test_recall = test_report_dict['weighted avg']['recall']
    weighted_test_f1 = test_report_dict['weighted avg']['f1-score']

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training SVM with PCA...

Weighted Average (Training) for SVM:
Precision: 0.72, Recall: 0.95, F1-Score: 0.82

Accuracy for SVM (Testing): 0.72

Classification Report for SVM (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.56      0.22      0.31        23
   Parkinson's       0.74      0.93      0.82        55

      accuracy                           0.72        78
     macro avg       0.65      0.57      0.57        78
  weighted avg       0.68      0.72      0.67        78


Confusion Matrix for SVM (Testing):
[[ 5 18]
 [ 4 51]]

Weighted Average (Testing) for SVM:
Precision: 0.68, Recall: 0.72, F1-Score: 0.67

Training MLP with PCA...


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perce


Weighted Average (Training) for MLP:
Precision: 0.74, Recall: 0.75, F1-Score: 0.74


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(



Accuracy for MLP (Testing): 0.71

Classification Report for MLP (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.48      0.49        23
   Parkinson's       0.79      0.80      0.79        55

      accuracy                           0.71        78
     macro avg       0.64      0.64      0.64        78
  weighted avg       0.70      0.71      0.70        78


Confusion Matrix for MLP (Testing):
[[11 12]
 [11 44]]

Weighted Average (Testing) for MLP:
Precision: 0.70, Recall: 0.71, F1-Score: 0.70

Training CatBoost with PCA...

Weighted Average (Training) for CatBoost:
Precision: 0.75, Recall: 0.91, F1-Score: 0.82

Accuracy for CatBoost (Testing): 0.81

Classification Report for CatBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.83      0.43      0.57        23
   Parkinson's       0.80      0.96      0.88        55

      accuracy                           0.81        78
     macro avg

PCA with smote

In [12]:
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier()
}

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with PCA and SMOTE...")

    fold_results = []  # To collect results for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_pca, y_train)):
        X_fold_train, X_fold_val = X_train_pca[train_idx], X_train_pca[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Apply SMOTE to the training fold
        X_fold_train_smote, y_fold_train_smote = smote.fit_resample(X_fold_train, y_fold_train)

        # Train the model
        model.fit(X_fold_train_smote, y_fold_train_smote)
        y_fold_pred = model.predict(X_fold_val)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the entire training set for testing
    model.fit(X_train_pca, y_train)

    # Get predictions for testing set
    y_test_pred = model.predict(X_test_pca)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions in the specified format
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set in the specified format
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    weighted_test_precision = test_report_dict['weighted avg']['precision']
    weighted_test_recall = test_report_dict['weighted avg']['recall']
    weighted_test_f1 = test_report_dict['weighted avg']['f1-score']

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training KNN with PCA and SMOTE...

Weighted Average (Training) for KNN:
Precision: 0.78, Recall: 0.64, F1-Score: 0.70

Accuracy for KNN (Testing): 0.71

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.48      0.49        23
   Parkinson's       0.79      0.80      0.79        55

      accuracy                           0.71        78
     macro avg       0.64      0.64      0.64        78
  weighted avg       0.70      0.71      0.70        78


Confusion Matrix for KNN (Testing):
[[11 12]
 [11 44]]

Weighted Average (Testing) for KNN:
Precision: 0.70, Recall: 0.71, F1-Score: 0.70

Training Naive Bayes with PCA and SMOTE...

Weighted Average (Training) for Naive Bayes:
Precision: 0.81, Recall: 0.65, F1-Score: 0.72

Accuracy for Naive Bayes (Testing): 0.73

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.58      0.30      

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1


Weighted Average (Training) for AdaBoost:
Precision: 0.77, Recall: 0.68, F1-Score: 0.72

Accuracy for AdaBoost (Testing): 0.71

Classification Report for AdaBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.43      0.47        23
   Parkinson's       0.78      0.82      0.80        55

      accuracy                           0.71        78
     macro avg       0.64      0.63      0.63        78
  weighted avg       0.69      0.71      0.70        78


Confusion Matrix for AdaBoost (Testing):
[[10 13]
 [10 45]]

Weighted Average (Testing) for AdaBoost:
Precision: 0.69, Recall: 0.71, F1-Score: 0.70

Training XGB with PCA and SMOTE...


/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



Weighted Average (Training) for XGB:
Precision: 0.75, Recall: 0.70, F1-Score: 0.72

Accuracy for XGB (Testing): 0.74

Classification Report for XGB (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.59      0.43      0.50        23
   Parkinson's       0.79      0.87      0.83        55

      accuracy                           0.74        78
     macro avg       0.69      0.65      0.66        78
  weighted avg       0.73      0.74      0.73        78


Confusion Matrix for XGB (Testing):
[[10 13]
 [ 7 48]]

Weighted Average (Testing) for XGB:
Precision: 0.73, Recall: 0.74, F1-Score: 0.73

Training Extra Trees with PCA and SMOTE...

Weighted Average (Training) for Extra Trees:
Precision: 0.76, Recall: 0.79, F1-Score: 0.77

Accuracy for Extra Trees (Testing): 0.81

Classification Report for Extra Trees (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.83      0.43      0.57        23
   Parkinson's       

SVM, MLP, cat and poly with smote

In [14]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

# Define models
models = {
    'SVM': SVC(kernel='rbf', C=1, gamma='scale', probability=True, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, activation='relu', solver='adam', random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': make_pipeline(PolynomialFeatures(degree=2), LogisticRegression(max_iter=1000)),
}

# Apply PCA to reduce dimensions
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Stratified K-Fold cross-validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# SMOTE for oversampling
smote = SMOTE(random_state=42)

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with PCA and SMOTE...")

    fold_results = []  # Collect results for each fold

    # Cross-validation loop
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_pca, y_train)):
        # Select appropriate dataset
        if name == 'CatBoost':  # CatBoost does not require PCA
            X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            X_train_final, X_test_final = X_train_scaled, X_test_scaled
        else:
            X_fold_train, X_fold_val = X_train_pca[train_idx], X_train_pca[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            X_train_final, X_test_final = X_train_pca, X_test_pca

        # Apply SMOTE
        X_fold_train_resampled, y_fold_train_resampled = smote.fit_resample(X_fold_train, y_fold_train)

        # Train model
        model.fit(X_fold_train_resampled, y_fold_train_resampled)
        y_fold_pred = model.predict(X_fold_val)

        # Store fold results
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Apply SMOTE to the full training set for testing
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_final, y_train)

    # Fit the model on the resampled training data
    model.fit(X_train_resampled, y_train_resampled)
    y_test_pred = model.predict(X_test_final)

    # Compute testing accuracy
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print classification report for the test set
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Print confusion matrix for the test set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Weighted averages for the test results
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    weighted_test_precision = test_report_dict['weighted avg']['precision']
    weighted_test_recall = test_report_dict['weighted avg']['recall']
    weighted_test_f1 = test_report_dict['weighted avg']['f1-score']

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training SVM with PCA and SMOTE...

Weighted Average (Training) for SVM:
Precision: 0.78, Recall: 0.75, F1-Score: 0.76

Accuracy for SVM (Testing): 0.65

Classification Report for SVM (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.44      0.65      0.53        23
   Parkinson's       0.82      0.65      0.73        55

      accuracy                           0.65        78
     macro avg       0.63      0.65      0.63        78
  weighted avg       0.71      0.65      0.67        78


Confusion Matrix for SVM (Testing):
[[15  8]
 [19 36]]

Weighted Average (Testing) for SVM:
Precision: 0.71, Recall: 0.65, F1-Score: 0.67

Training MLP with PCA and SMOTE...


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perce


Weighted Average (Training) for MLP:
Precision: 0.75, Recall: 0.73, F1-Score: 0.73


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(



Accuracy for MLP (Testing): 0.72

Classification Report for MLP (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.52      0.57      0.54        23
   Parkinson's       0.81      0.78      0.80        55

      accuracy                           0.72        78
     macro avg       0.67      0.67      0.67        78
  weighted avg       0.73      0.72      0.72        78


Confusion Matrix for MLP (Testing):
[[13 10]
 [12 43]]

Weighted Average (Testing) for MLP:
Precision: 0.73, Recall: 0.72, F1-Score: 0.72

Training CatBoost with PCA and SMOTE...

Weighted Average (Training) for CatBoost:
Precision: 0.79, Recall: 0.73, F1-Score: 0.76

Accuracy for CatBoost (Testing): 0.72

Classification Report for CatBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.52      0.70      0.59        23
   Parkinson's       0.85      0.73      0.78        55

      accuracy                           0.72        78
    

PCA smote 0.5

In [4]:
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier()
}

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Initialize SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)

# Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with PCA and SMOTE...")

    fold_results = []  # To collect results for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_pca, y_train)):
        X_fold_train, X_fold_val = X_train_pca[train_idx], X_train_pca[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Apply SMOTE to the training fold
        X_fold_train_smote, y_fold_train_smote = smote.fit_resample(X_fold_train, y_fold_train)

        # Train the model
        model.fit(X_fold_train_smote, y_fold_train_smote)
        y_fold_pred = model.predict(X_fold_val)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the entire training set for testing
    model.fit(X_train_pca, y_train)

    # Get predictions for testing set
    y_test_pred = model.predict(X_test_pca)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions in the specified format
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set in the specified format
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    weighted_test_precision = test_report_dict['weighted avg']['precision']
    weighted_test_recall = test_report_dict['weighted avg']['recall']
    weighted_test_f1 = test_report_dict['weighted avg']['f1-score']

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")

/usr/local/lib/python3.10/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)



Training KNN with PCA and SMOTE...

Weighted Average (Training) for KNN:
Precision: 0.77, Recall: 0.80, F1-Score: 0.78

Accuracy for KNN (Testing): 0.71

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.48      0.49        23
   Parkinson's       0.79      0.80      0.79        55

      accuracy                           0.71        78
     macro avg       0.64      0.64      0.64        78
  weighted avg       0.70      0.71      0.70        78


Confusion Matrix for KNN (Testing):
[[11 12]
 [11 44]]

Weighted Average (Testing) for KNN:
Precision: 0.70, Recall: 0.71, F1-Score: 0.70

Training Naive Bayes with PCA and SMOTE...

Weighted Average (Training) for Naive Bayes:
Precision: 0.78, Recall: 0.85, F1-Score: 0.81

Accuracy for Naive Bayes (Testing): 0.73

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.58      0.30      

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1


Weighted Average (Training) for AdaBoost:
Precision: 0.76, Recall: 0.79, F1-Score: 0.77

Accuracy for AdaBoost (Testing): 0.71

Classification Report for AdaBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.43      0.47        23
   Parkinson's       0.78      0.82      0.80        55

      accuracy                           0.71        78
     macro avg       0.64      0.63      0.63        78
  weighted avg       0.69      0.71      0.70        78


Confusion Matrix for AdaBoost (Testing):
[[10 13]
 [10 45]]

Weighted Average (Testing) for AdaBoost:
Precision: 0.69, Recall: 0.71, F1-Score: 0.70

Training XGB with PCA and SMOTE...


/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(



Weighted Average (Training) for XGB:
Precision: 0.77, Recall: 0.81, F1-Score: 0.79

Accuracy for XGB (Testing): 0.74

Classification Report for XGB (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.59      0.43      0.50        23
   Parkinson's       0.79      0.87      0.83        55

      accuracy                           0.74        78
     macro avg       0.69      0.65      0.66        78
  weighted avg       0.73      0.74      0.73        78


Confusion Matrix for XGB (Testing):
[[10 13]
 [ 7 48]]

Weighted Average (Testing) for XGB:
Precision: 0.73, Recall: 0.74, F1-Score: 0.73

Training Extra Trees with PCA and SMOTE...

Weighted Average (Training) for Extra Trees:
Precision: 0.76, Recall: 0.89, F1-Score: 0.82

Accuracy for Extra Trees (Testing): 0.76

Classification Report for Extra Trees (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.64      0.39      0.49        23
   Parkinson's       

SVM with 0.5 smote

In [15]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

# Define models
models = {
    'SVM': SVC(kernel='rbf', C=1, gamma='scale', probability=True, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, activation='relu', solver='adam', random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': make_pipeline(PolynomialFeatures(degree=2), LogisticRegression(max_iter=1000)),
}

# Apply PCA to reduce dimensions
pca = PCA(n_components=6)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Stratified K-Fold cross-validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# SMOTE for oversampling
smote = SMOTE(sampling_strategy=0.5, random_state=42)

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with PCA and SMOTE...")

    fold_results = []  # Collect results for each fold

    # Cross-validation loop
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_pca, y_train)):
        # Select appropriate dataset
        if name == 'CatBoost':  # CatBoost does not require PCA
            X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            X_train_final, X_test_final = X_train_scaled, X_test_scaled
        else:
            X_fold_train, X_fold_val = X_train_pca[train_idx], X_train_pca[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            X_train_final, X_test_final = X_train_pca, X_test_pca

        # Apply SMOTE
        X_fold_train_resampled, y_fold_train_resampled = smote.fit_resample(X_fold_train, y_fold_train)

        # Train model
        model.fit(X_fold_train_resampled, y_fold_train_resampled)
        y_fold_pred = model.predict(X_fold_val)

        # Store fold results
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Apply SMOTE to the full training set for testing
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_final, y_train)

    # Fit the model on the resampled training data
    model.fit(X_train_resampled, y_train_resampled)
    y_test_pred = model.predict(X_test_final)

    # Compute testing accuracy
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print classification report for the test set
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Print confusion matrix for the test set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Weighted averages for the test results
    test_report_dict = classification_report(y_test, y_test_pred, output_dict=True, zero_division=0)
    weighted_test_precision = test_report_dict['weighted avg']['precision']
    weighted_test_recall = test_report_dict['weighted avg']['recall']
    weighted_test_f1 = test_report_dict['weighted avg']['f1-score']

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")


Training SVM with PCA and SMOTE...

Weighted Average (Training) for SVM:
Precision: 0.76, Recall: 0.89, F1-Score: 0.82

Accuracy for SVM (Testing): 0.65

Classification Report for SVM (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.41      0.39      0.40        23
   Parkinson's       0.75      0.76      0.76        55

      accuracy                           0.65        78
     macro avg       0.58      0.58      0.58        78
  weighted avg       0.65      0.65      0.65        78


Confusion Matrix for SVM (Testing):
[[ 9 14]
 [13 42]]

Weighted Average (Testing) for SVM:
Precision: 0.65, Recall: 0.65, F1-Score: 0.65

Training MLP with PCA and SMOTE...


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perce


Weighted Average (Training) for MLP:
Precision: 0.74, Recall: 0.75, F1-Score: 0.74


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(



Accuracy for MLP (Testing): 0.72

Classification Report for MLP (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.52      0.52      0.52        23
   Parkinson's       0.80      0.80      0.80        55

      accuracy                           0.72        78
     macro avg       0.66      0.66      0.66        78
  weighted avg       0.72      0.72      0.72        78


Confusion Matrix for MLP (Testing):
[[12 11]
 [11 44]]

Weighted Average (Testing) for MLP:
Precision: 0.72, Recall: 0.72, F1-Score: 0.72

Training CatBoost with PCA and SMOTE...

Weighted Average (Training) for CatBoost:
Precision: 0.76, Recall: 0.89, F1-Score: 0.82

Accuracy for CatBoost (Testing): 0.79

Classification Report for CatBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.73      0.48      0.58        23
   Parkinson's       0.81      0.93      0.86        55

      accuracy                           0.79        78
    

ICA

In [19]:
from sklearn.decomposition import FastICA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Initialize ICA with a specified number of components (for example, 5)
ica = FastICA(n_components=10, random_state=42)

# Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier()
}

# Initialize StandardScaler for scaling the data
scaler = StandardScaler()

# Scale the data (fit and transform the features)
X_scaled = scaler.fit_transform(X)  # X is your feature matrix

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with ICA...")

    fold_results = []  # To collect results for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_scaled, y)):
        X_fold_train, X_fold_val = X_scaled[train_idx], X_scaled[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

        # Apply ICA transformation
        X_fold_train_ica = ica.fit_transform(X_fold_train)  # Apply ICA to the training fold
        X_fold_val_ica = ica.transform(X_fold_val)  # Apply ICA to the validation fold

        # Train the model
        model.fit(X_fold_train_ica, y_fold_train)
        y_fold_pred = model.predict(X_fold_val_ica)

        # Only store the report for the last fold (10th fold)
        if fold_idx == 9:
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))

            # Display confusion matrix for 10th fold
            conf_matrix = confusion_matrix(y_fold_val, y_fold_pred)
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(conf_matrix)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the entire training set for testing
    X_train_ica = ica.fit_transform(X_scaled)  # Apply ICA to the entire training set
    model.fit(X_train_ica, y)

    # Get predictions for testing set
    X_test_ica = ica.transform(X_test)  # Apply ICA to the testing set
    y_test_pred = model.predict(X_test_ica)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training KNN with ICA...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.55      0.55      0.55        11
   Parkinson's       0.82      0.82      0.82        28

      accuracy                           0.74        39
     macro avg       0.68      0.68      0.68        39
  weighted avg       0.74      0.74      0.74        39


Confusion Matrix for KNN (10th Fold):
[[ 6  5]
 [ 5 23]]

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.79, F1-Score: 0.77

Accuracy for KNN (Testing): 0.77

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.78      0.30      0.44        23
   Parkinson's       0.77      0.96      0.85        55

      accuracy                           0.77        78
     macro avg       0.77      0.63      0.65        78
  weighted avg       0.77      0.77      0.73        78


Confusion Matrix for KNN (Testing

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Classification Report for Naive Bayes (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.60      0.27      0.38        11
   Parkinson's       0.76      0.93      0.84        28

      accuracy                           0.74        39
     macro avg       0.68      0.60      0.61        39
  weighted avg       0.72      0.74      0.71        39


Confusion Matrix for Naive Bayes (10th Fold):
[[ 3  8]
 [ 2 26]]

Weighted Average (Training) for Naive Bayes:
Precision: 0.76, Recall: 0.88, F1-Score: 0.82

Accuracy for Naive Bayes (Testing): 0.71

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix fo

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Classification Report for Logistic Regression (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.75      0.27      0.40        11
   Parkinson's       0.77      0.96      0.86        28

      accuracy                           0.77        39
     macro avg       0.76      0.62      0.63        39
  weighted avg       0.77      0.77      0.73        39


Confusion Matrix for Logistic Regression (10th Fold):
[[ 3  8]
 [ 1 27]]

Weighted Average (Training) for Logistic Regression:
Precision: 0.77, Recall: 0.92, F1-Score: 0.84

Accuracy for Logistic Regression (Testing): 0.71

Classification Report for Logistic Regression (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71  

...

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Accuracy for XGB (Testing): 0.72

Classification Report for XGB (Testing):
                precision    recall  f1-score   support

No Parkinson's       1.00      0.04      0.08        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.72        78
     macro avg       0.86      0.52      0.46        78
  weighted avg       0.80      0.72      0.61        78


Confusion Matrix for XGB (Testing):
[[ 1 22]
 [ 0 55]]

Weighted Average (Testing) for XGB:
Precision: 0.00, Recall: 0.00, F1-Score: 0.00

Training Extra Trees with ICA...

Classification Report for Extra Trees (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.27      0.35        11
   Parkinson's       0.76      0.89      0.82        28

      accuracy                           0.72        39
     macro avg       0.63      0.58      0.59        39
  weighted avg       0.68      0.72      0.69        39


Confusion Matrix for

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Accuracy for Extra Trees (Testing): 0.71

Classification Report for Extra Trees (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix for Extra Trees (Testing):
[[ 0 23]
 [ 0 55]]

Weighted Average (Testing) for Extra Trees:
Precision: 0.00, Recall: 0.00, F1-Score: 0.00

Training LightGBM with ICA...
[LightGBM] [Info] Number of positive: 249, number of negative: 102
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000080 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1160
[LightGBM] [Info] Number of data points in the train set: 351, number of used features: 10
[LightGBM] [Info] [binary:Boo

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(


ICA with SVM, MLP, Cat and polynomial

In [20]:
from sklearn.decomposition import FastICA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np

# Initialize ICA with a specified number of components
ica = FastICA(n_components=10, random_state=42)

# Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define classifiers
models = {
    'SVM': SVC(kernel='rbf', C=1, gamma='scale', probability=True, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(100,), max_iter=2000, activation='relu', solver='adam', random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': make_pipeline(PolynomialFeatures(degree=2), LogisticRegression(max_iter=1000))
}

# Initialize StandardScaler for scaling the data
scaler = StandardScaler()

# Scale the data (fit and transform the features)
X_scaled = scaler.fit_transform(X)  # X is your feature matrix

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with ICA...")

    fold_results = []  # To collect results for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_scaled, y)):
        X_fold_train, X_fold_val = X_scaled[train_idx], X_scaled[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

        # Apply ICA transformation
        X_fold_train_ica = ica.fit_transform(X_fold_train)  # Apply ICA to the training fold
        X_fold_val_ica = ica.transform(X_fold_val)  # Apply ICA to the validation fold

        # Train the model
        model.fit(X_fold_train_ica, y_fold_train)
        y_fold_pred = model.predict(X_fold_val_ica)

        # Only store the report for the last fold (10th fold)
        if fold_idx == 9:
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))

            # Display confusion matrix for 10th fold
            conf_matrix = confusion_matrix(y_fold_val, y_fold_pred)
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(conf_matrix)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the entire training set for testing
    X_train_ica = ica.fit_transform(X_scaled)  # Apply ICA to the entire training set
    model.fit(X_train_ica, y)

    # Get predictions for testing set
    X_test_ica = ica.transform(X_test)  # Apply ICA to the testing set
    y_test_pred = model.predict(X_test_ica)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training SVM with ICA...

Classification Report for SVM (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.18      0.27        11
   Parkinson's       0.74      0.93      0.83        28

      accuracy                           0.72        39
     macro avg       0.62      0.56      0.55        39
  weighted avg       0.67      0.72      0.67        39


Confusion Matrix for SVM (10th Fold):
[[ 2  9]
 [ 2 26]]

Weighted Average (Training) for SVM:
Precision: 0.74, Recall: 0.91, F1-Score: 0.82

Accuracy for SVM (Testing): 0.71

Classification Report for SVM (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix for SVM (Testing

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Classification Report for MLP (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.23      0.27      0.25        11
   Parkinson's       0.69      0.64      0.67        28

      accuracy                           0.54        39
     macro avg       0.46      0.46      0.46        39
  weighted avg       0.56      0.54      0.55        39


Confusion Matrix for MLP (10th Fold):
[[ 3  8]
 [10 18]]

Weighted Average (Training) for MLP:
Precision: 0.76, Recall: 0.79, F1-Score: 0.77


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Accuracy for MLP (Testing): 0.81

Classification Report for MLP (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.79      0.48      0.59        23
   Parkinson's       0.81      0.95      0.87        55

      accuracy                           0.81        78
     macro avg       0.80      0.71      0.73        78
  weighted avg       0.80      0.81      0.79        78


Confusion Matrix for MLP (Testing):
[[11 12]
 [ 3 52]]

Weighted Average (Testing) for MLP:
Precision: 0.00, Recall: 0.00, F1-Score: 0.00

Training CatBoost with ICA...

Classification Report for CatBoost (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.60      0.27      0.38        11
   Parkinson's       0.76      0.93      0.84        28

      accuracy                           0.74        39
     macro avg       0.68      0.60      0.61        39
  weighted avg       0.72      0.74      0.71        39


Confusion Matrix for CatBo

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(



Accuracy for CatBoost (Testing): 0.71

Classification Report for CatBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix for CatBoost (Testing):
[[ 0 23]
 [ 0 55]]

Weighted Average (Testing) for CatBoost:
Precision: 0.00, Recall: 0.00, F1-Score: 0.00

Training Polynomial Regression with ICA...

Classification Report for Polynomial Regression (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.30      0.27      0.29        11
   Parkinson's       0.72      0.75      0.74        28

      accuracy                           0.62        39
     macro avg       0.51      0.51      0.51        39
  weighted avg       0.60      0.62   

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but FastICA was fitted without feature names
  warnings.warn(


ICA with smote

In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import FastICA
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

# Assuming you have loaded your data into X and y
# X is the feature matrix, y is the target variable

# Step 1: Stratified train-test split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Step 2: Apply ICA (Independent Component Analysis) for dimensionality reduction
ica = FastICA(n_components=10, random_state=42)  # Example, adjust components as needed
X_train_ica = ica.fit_transform(X_train)
X_test_ica = ica.transform(X_test)

# Step 3: Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ica)
X_test_scaled = scaler.transform(X_test_ica)

# Step 4: Apply SMOTE for balancing classes in the training set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Step 5: Define classifiers (no hyperparameter tuning)
models = {
    'KNeighborsClassifier': KNeighborsClassifier(),
    'GaussianNB': GaussianNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'RandomForestClassifier': RandomForestClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'AdaBoostClassifier': AdaBoostClassifier(),
    'ExtraTreesClassifier': ExtraTreesClassifier(),
    'XGBClassifier': XGBClassifier(),
    'LGBMClassifier': LGBMClassifier(),
    # New models
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Step 6: Evaluate each classifier
classification_reports = {}

for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")

    # Create a pipeline for the model
    pipeline = Pipeline([
        ('model', model)
    ])

    # Fit the model after resampling with SMOTE
    pipeline.fit(X_train_smote, y_train_smote)

    # Evaluate on the test set
    y_pred = pipeline.predict(X_test_scaled)

    # Store the classification report for this model
    classification_reports[model_name] = classification_report(y_test, y_pred)

    # Calculate Weighted Average Precision, Recall, F1-Score during training
    precision_weighted = np.mean(cross_val_score(pipeline, X_train_smote, y_train_smote, cv=StratifiedKFold(n_splits=10), scoring='precision_weighted'))
    recall_weighted = np.mean(cross_val_score(pipeline, X_train_smote, y_train_smote, cv=StratifiedKFold(n_splits=10), scoring='recall_weighted'))
    f1_weighted = np.mean(cross_val_score(pipeline, X_train_smote, y_train_smote, cv=StratifiedKFold(n_splits=10), scoring='f1_weighted'))

    # Print classification report for the model
    print(f"Classification Report for {model_name}:\n{classification_reports[model_name]}")
    print(f"Weighted Average Precision for {model_name} (Training): {precision_weighted:.4f}")
    print(f"Weighted Average Recall for {model_name} (Training): {recall_weighted:.4f}")
    print(f"Weighted Average F1-Score for {model_name} (Training): {f1_weighted:.4f}")

# Optional: print all classification reports
for model_name, report in classification_reports.items():
    print(f"\n{model_name} Classification Report:\n{report}")


Training and evaluating KNeighborsClassifier...
Classification Report for KNeighborsClassifier:
              precision    recall  f1-score   support

           0       0.38      0.70      0.49        23
           1       0.81      0.53      0.64        55

    accuracy                           0.58        78
   macro avg       0.59      0.61      0.56        78
weighted avg       0.68      0.58      0.59        78

Weighted Average Precision for KNeighborsClassifier (Training): 0.7828
Weighted Average Recall for KNeighborsClassifier (Training): 0.7469
Weighted Average F1-Score for KNeighborsClassifier (Training): 0.7377

Training and evaluating GaussianNB...
Classification Report for GaussianNB:
              precision    recall  f1-score   support

           0       0.47      0.70      0.56        23
           1       0.84      0.67      0.75        55

    accuracy                           0.68        78
   macro avg       0.66      0.68      0.65        78
weighted avg      

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1

Classification Report for AdaBoostClassifier:
              precision    recall  f1-score   support

           0       0.54      0.61      0.57        23
           1       0.83      0.78      0.80        55

    accuracy                           0.73        78
   macro avg       0.68      0.70      0.69        78
weighted avg       0.74      0.73      0.74        78

Weighted Average Precision for AdaBoostClassifier (Training): 0.6714
Weighted Average Recall for AdaBoostClassifier (Training): 0.6681
Weighted Average F1-Score for AdaBoostClassifier (Training): 0.6664

Training and evaluating ExtraTreesClassifier...
Classification Report for ExtraTreesClassifier:
              precision    recall  f1-score   support

           0       0.63      0.74      0.68        23
           1       0.88      0.82      0.85        55

    accuracy                           0.79        78
   macro avg       0.76      0.78      0.76        78
weighted avg       0.81      0.79      0.80        78



ICA with smote 0.5

In [22]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import FastICA
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

# Assuming you have loaded your data into X and y
# X is the feature matrix, y is the target variable

# Step 1: Stratified train-test split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Step 2: Apply ICA (Independent Component Analysis) for dimensionality reduction
ica = FastICA(n_components=10, random_state=42)  # Example, adjust components as needed
X_train_ica = ica.fit_transform(X_train)
X_test_ica = ica.transform(X_test)

# Step 3: Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ica)
X_test_scaled = scaler.transform(X_test_ica)

# Step 4: Apply SMOTE for balancing classes in the training set
smote = SMOTE(sampling_strategy=0.5,random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Step 5: Define classifiers (no hyperparameter tuning)
models = {
    'KNeighborsClassifier': KNeighborsClassifier(),
    'GaussianNB': GaussianNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'RandomForestClassifier': RandomForestClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'AdaBoostClassifier': AdaBoostClassifier(),
    'ExtraTreesClassifier': ExtraTreesClassifier(),
    'XGBClassifier': XGBClassifier(),
    'LGBMClassifier': LGBMClassifier(),
    # New models
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Step 6: Evaluate each classifier
classification_reports = {}

for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")

    # Create a pipeline for the model
    pipeline = Pipeline([
        ('model', model)
    ])

    # Fit the model after resampling with SMOTE
    pipeline.fit(X_train_smote, y_train_smote)

    # Evaluate on the test set
    y_pred = pipeline.predict(X_test_scaled)

    # Store the classification report for this model
    classification_reports[model_name] = classification_report(y_test, y_pred)

    # Calculate Weighted Average Precision, Recall, F1-Score during training
    precision_weighted = np.mean(cross_val_score(pipeline, X_train_smote, y_train_smote, cv=StratifiedKFold(n_splits=10), scoring='precision_weighted'))
    recall_weighted = np.mean(cross_val_score(pipeline, X_train_smote, y_train_smote, cv=StratifiedKFold(n_splits=10), scoring='recall_weighted'))
    f1_weighted = np.mean(cross_val_score(pipeline, X_train_smote, y_train_smote, cv=StratifiedKFold(n_splits=10), scoring='f1_weighted'))

    # Print classification report for the model
    print(f"Classification Report for {model_name}:\n{classification_reports[model_name]}")
    print(f"Weighted Average Precision for {model_name} (Training): {precision_weighted:.4f}")
    print(f"Weighted Average Recall for {model_name} (Training): {recall_weighted:.4f}")
    print(f"Weighted Average F1-Score for {model_name} (Training): {f1_weighted:.4f}")

# Optional: print all classification reports
for model_name, report in classification_reports.items():
    print(f"\n{model_name} Classification Report:\n{report}")


Training and evaluating KNeighborsClassifier...
Classification Report for KNeighborsClassifier:
              precision    recall  f1-score   support

           0       0.42      0.48      0.45        23
           1       0.77      0.73      0.75        55

    accuracy                           0.65        78
   macro avg       0.60      0.60      0.60        78
weighted avg       0.67      0.65      0.66        78

Weighted Average Precision for KNeighborsClassifier (Training): 0.6546
Weighted Average Recall for KNeighborsClassifier (Training): 0.6432
Weighted Average F1-Score for KNeighborsClassifier (Training): 0.6309

Training and evaluating GaussianNB...
Classification Report for GaussianNB:
              precision    recall  f1-score   support

           0       0.58      0.61      0.60        23
           1       0.83      0.82      0.83        55

    accuracy                           0.76        78
   macro avg       0.71      0.71      0.71        78
weighted avg      

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1

Classification Report for AdaBoostClassifier:
              precision    recall  f1-score   support

           0       0.64      0.61      0.62        23
           1       0.84      0.85      0.85        55

    accuracy                           0.78        78
   macro avg       0.74      0.73      0.73        78
weighted avg       0.78      0.78      0.78        78

Weighted Average Precision for AdaBoostClassifier (Training): 0.6668
Weighted Average Recall for AdaBoostClassifier (Training): 0.6735
Weighted Average F1-Score for AdaBoostClassifier (Training): 0.6651

Training and evaluating ExtraTreesClassifier...
Classification Report for ExtraTreesClassifier:
              precision    recall  f1-score   support

           0       0.71      0.43      0.54        23
           1       0.80      0.93      0.86        55

    accuracy                           0.78        78
   macro avg       0.76      0.68      0.70        78
weighted avg       0.77      0.78      0.76        78



Annova

In [30]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define the number of features you want to select with ANOVA
k_best = 8 # You can change this to the number of features you want to keep

# Initialize Stratified K-Fold Cross-Validation
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Initialize StandardScaler for scaling the data
scaler = StandardScaler()

# Scale the data (fit and transform the features)
X_scaled = scaler.fit_transform(X)  # X is your feature matrix

# Apply ANOVA for feature selection (selecting top 5 features as an example)
anova = SelectKBest(f_classif, k=k_best)
X_anova = anova.fit_transform(X_scaled, y)  # Selecting the best features based on ANOVA

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with ANOVA...")

    fold_results = []  # To collect results for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_anova, y)):
        X_fold_train, X_fold_val = X_anova[train_idx], X_anova[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

        # Train the model
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Only store the report for the last fold (10th fold)
        if fold_idx == 9:
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))

            # Display confusion matrix for 10th fold
            conf_matrix = confusion_matrix(y_fold_val, y_fold_pred)
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(conf_matrix)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Fit the model on the entire training set for testing
    model.fit(X_anova, y)

    # Get predictions for testing set
    X_test_anova = anova.transform(X_test)  # Apply the ANOVA transformation to the testing set
    y_test_pred = model.predict(X_test_anova)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training KNN with ANOVA...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.36      0.42        11
   Parkinson's       0.77      0.86      0.81        28

      accuracy                           0.72        39
     macro avg       0.64      0.61      0.62        39
  weighted avg       0.70      0.72      0.70        39


Confusion Matrix for KNN (10th Fold):
[[ 4  7]
 [ 4 24]]

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.83, F1-Score: 0.79

Accuracy for KNN (Testing): 0.71

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.04      0.08        23
   Parkinson's       0.71      0.98      0.82        55

      accuracy                           0.71        78
     macro avg       0.61      0.51      0.45        78
  weighted avg       0.65      0.71      0.60        78


Confusion Matrix for KNN (Testi

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(



Classification Report for Naive Bayes (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.55      0.55      0.55        11
   Parkinson's       0.82      0.82      0.82        28

      accuracy                           0.74        39
     macro avg       0.68      0.68      0.68        39
  weighted avg       0.74      0.74      0.74        39


Confusion Matrix for Naive Bayes (10th Fold):
[[ 6  5]
 [ 5 23]]

Weighted Average (Training) for Naive Bayes:
Precision: 0.81, Recall: 0.80, F1-Score: 0.80

Accuracy for Naive Bayes (Testing): 0.71

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix fo

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(


                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix for Logistic Regression (Testing):
[[ 0 23]
 [ 0 55]]

Weighted Average (Testing) for Logistic Regression:
Precision: 0.00, Recall: 0.00, F1-Score: 0.00

Training Decision Tree with ANOVA...

Classification Report for Decision Tree (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.67      0.55      0.60        11
   Parkinson's       0.83      0.89      0.86        28

      accuracy                           0.79        39
     macro avg       0.75      0.72      0.73        39
  weighted avg       0.79      0.79      0.79        39


Confusion Matrix for Decision Tree (10th Fold):
[[ 6  5]
 

...

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(



Accuracy for MLP (Testing): 0.71

Classification Report for MLP (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50      0.71      0.58        78


Confusion Matrix for MLP (Testing):
[[ 0 23]
 [ 0 55]]

Weighted Average (Testing) for MLP:
Precision: 0.00, Recall: 0.00, F1-Score: 0.00

Training CatBoost with ANOVA...

Classification Report for CatBoost (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.57      0.36      0.44        11
   Parkinson's       0.78      0.89      0.83        28

      accuracy                           0.74        39
     macro avg       0.68      0.63      0.64        39
  weighted avg       0.72      0.74      0.72        39


Confusion Matrix for Cat

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(



Classification Report for Polynomial Regression (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.57      0.36      0.44        11
   Parkinson's       0.78      0.89      0.83        28

      accuracy                           0.74        39
     macro avg       0.68      0.63      0.64        39
  weighted avg       0.72      0.74      0.72        39


Confusion Matrix for Polynomial Regression (10th Fold):
[[ 4  7]
 [ 3 25]]

Weighted Average (Training) for Polynomial Regression:
Precision: 0.83, Recall: 0.87, F1-Score: 0.85

Accuracy for Polynomial Regression (Testing): 0.71

Classification Report for Polynomial Regression (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.00      0.00      0.00        23
   Parkinson's       0.71      1.00      0.83        55

      accuracy                           0.71        78
     macro avg       0.35      0.50      0.41        78
  weighted avg       0.50  

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but SelectKBest was fitted without feature names
  warnings.warn(


Annova with smote

In [34]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE  # Import SMOTE
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif

# Define the number of features you want to select with ANOVA
k_best = 8  # You can change this to the number of features you want to keep

# Initialize StandardScaler for scaling the data
scaler = StandardScaler()

# Scale the data (fit and transform the features)
X_scaled = scaler.fit_transform(X)  # Scale the entire feature matrix
X_test_scaled = scaler.transform(X_test)  # Scale the test data

# Apply ANOVA for feature selection (selecting top 8 features as an example)
anova = SelectKBest(f_classif, k=k_best)
X_anova = anova.fit_transform(X_scaled, y)  # Selecting the best features based on ANOVA
X_test_anova = anova.transform(X_test_scaled)  # Apply the same transformation to the test set

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Initialize StratifiedKFold with 10 splits
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with 10-Fold Cross-Validation and SMOTE...")

    fold_results = []  # To collect results for each fold
    accuracy_scores = []  # To collect accuracy scores for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_anova, y)):
        X_fold_train, X_fold_val = X_anova[train_idx], X_anova[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

        # Apply SMOTE to the training fold
        X_fold_train_smote, y_fold_train_smote = smote.fit_resample(X_fold_train, y_fold_train)

        # Train the model
        model.fit(X_fold_train_smote, y_fold_train_smote)
        y_fold_pred = model.predict(X_fold_val)

        # Only store the report for the last fold (10th fold)
        if fold_idx == 9:
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))

            # Display confusion matrix for 10th fold
            conf_matrix = confusion_matrix(y_fold_val, y_fold_pred)
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(conf_matrix)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

        # Collect accuracy scores for this fold
        accuracy_scores.append(accuracy_score(y_fold_val, y_fold_pred))

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Print the average accuracy score for cross-validation
    print(f"\nAverage Accuracy for {name} (Training) across folds: {np.mean(accuracy_scores):.2f}")

    # Test the model on the test set
    y_test_pred = model.predict(X_test_anova)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training KNN with 10-Fold Cross-Validation and SMOTE...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.43      0.27      0.33        11
   Parkinson's       0.75      0.86      0.80        28

      accuracy                           0.69        39
     macro avg       0.59      0.56      0.57        39
  weighted avg       0.66      0.69      0.67        39


Confusion Matrix for KNN (10th Fold):
[[ 3  8]
 [ 4 24]]

Weighted Average (Training) for KNN:
Precision: 0.80, Recall: 0.78, F1-Score: 0.79

Average Accuracy for KNN (Training) across folds: 0.71

Accuracy for KNN (Testing): 0.77

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.63      0.52      0.57        23
   Parkinson's       0.81      0.87      0.84        55

      accuracy                           0.77        78
     macro avg       0.72      0.70      0.71        78
  we

Annova with 0.5 smote

In [35]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE  # Import SMOTE
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif

# Define the number of features you want to select with ANOVA
k_best = 8  # You can change this to the number of features you want to keep

# Initialize StandardScaler for scaling the data
scaler = StandardScaler()

# Scale the data (fit and transform the features)
X_scaled = scaler.fit_transform(X)  # Scale the entire feature matrix
X_test_scaled = scaler.transform(X_test)  # Scale the test data

# Apply ANOVA for feature selection (selecting top 8 features as an example)
anova = SelectKBest(f_classif, k=k_best)
X_anova = anova.fit_transform(X_scaled, y)  # Selecting the best features based on ANOVA
X_test_anova = anova.transform(X_test_scaled)  # Apply the same transformation to the test set

# Initialize SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)

# Initialize StratifiedKFold with 10 splits
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Iterate over each model
for name, model in models.items():
    print(f"\nTraining {name} with 10-Fold Cross-Validation and SMOTE...")

    fold_results = []  # To collect results for each fold
    accuracy_scores = []  # To collect accuracy scores for each fold

    # Cross-validated predictions
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_anova, y)):
        X_fold_train, X_fold_val = X_anova[train_idx], X_anova[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

        # Apply SMOTE to the training fold
        X_fold_train_smote, y_fold_train_smote = smote.fit_resample(X_fold_train, y_fold_train)

        # Train the model
        model.fit(X_fold_train_smote, y_fold_train_smote)
        y_fold_pred = model.predict(X_fold_val)

        # Only store the report for the last fold (10th fold)
        if fold_idx == 9:
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))

            # Display confusion matrix for 10th fold
            conf_matrix = confusion_matrix(y_fold_val, y_fold_pred)
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(conf_matrix)

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

        # Collect accuracy scores for this fold
        accuracy_scores.append(accuracy_score(y_fold_val, y_fold_pred))

    # Calculate weighted averages for cross-validation results
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Print the average accuracy score for cross-validation
    print(f"\nAverage Accuracy for {name} (Training) across folds: {np.mean(accuracy_scores):.2f}")

    # Test the model on the test set
    y_test_pred = model.predict(X_test_anova)

    # Compute accuracy for testing predictions
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    # Print the classification report for testing predictions
    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for testing set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")


Training KNN with 10-Fold Cross-Validation and SMOTE...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.36      0.42        11
   Parkinson's       0.77      0.86      0.81        28

      accuracy                           0.72        39
     macro avg       0.64      0.61      0.62        39
  weighted avg       0.70      0.72      0.70        39


Confusion Matrix for KNN (10th Fold):
[[ 4  7]
 [ 4 24]]

Weighted Average (Training) for KNN:
Precision: 0.77, Recall: 0.83, F1-Score: 0.80

Average Accuracy for KNN (Training) across folds: 0.70

Accuracy for KNN (Testing): 0.81

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.68      0.65      0.67        23
   Parkinson's       0.86      0.87      0.86        55

      accuracy                           0.81        78
     macro avg       0.77      0.76      0.77        78
  we

RFE

In [36]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Initialize RFE (Recursive Feature Elimination) on the training set only
rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=5)
X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)  # Apply RFE transformation to the test set as well

# Initialize Stratified K-Fold Cross-Validation on the training set
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Loop through each model
for name, model in models.items():
    print(f"\nTraining {name} with RFE (No SMOTE)...")

    fold_results = []  # To collect cross-validation results for each fold

    # Perform Stratified K-Fold Cross-Validation
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_rfe, y_train)):
        X_fold_train, X_fold_val = X_train_rfe[train_idx], X_train_rfe[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train the model on the current training fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        if fold_idx == 9:  # Display report for the last fold
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(confusion_matrix(y_fold_val, y_fold_pred))

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Train the model on the entire training set and evaluate on the test set
    model.fit(X_train_rfe, y_train)

    # Predict on the test set
    y_test_pred = model.predict(X_test_rfe)

    # Test set metrics
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for the test set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training KNN with RFE (No SMOTE)...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.33      0.22      0.27         9
   Parkinson's       0.72      0.82      0.77        22

      accuracy                           0.65        31
     macro avg       0.53      0.52      0.52        31
  weighted avg       0.61      0.65      0.62        31


Confusion Matrix for KNN (10th Fold):
[[ 2  7]
 [ 4 18]]

Weighted Average (Training) for KNN:
Precision: 0.78, Recall: 0.76, F1-Score: 0.77

Accuracy for KNN (Testing): 0.76

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.57      0.70      0.63        23
   Parkinson's       0.86      0.78      0.82        55

      accuracy                           0.76        78
     macro avg       0.72      0.74      0.72        78
  weighted avg       0.77      0.76      0.76        78


Confusion Matrix for K

RFE with Smote

In [37]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Initialize RFE (Recursive Feature Elimination) on the training set only
rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=5)
X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)  # Apply RFE transformation to the test set as well

# Initialize SMOTE for oversampling
smote = SMOTE(random_state=42)

# Initialize Stratified K-Fold Cross-Validation on the training set
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Loop through each model
for name, model in models.items():
    print(f"\nTraining {name} with RFE and SMOTE...")

    fold_results = []  # To collect cross-validation results for each fold

    # Perform Stratified K-Fold Cross-Validation
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_rfe, y_train)):
        X_fold_train, X_fold_val = X_train_rfe[train_idx], X_train_rfe[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Apply SMOTE to the training data of the current fold
        X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)

        # Train the model on the resampled data
        model.fit(X_resampled, y_resampled)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        if fold_idx == 9:  # Display report for the last fold
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(confusion_matrix(y_fold_val, y_fold_pred))

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Train on the entire training set with SMOTE and evaluate on the test set
    X_resampled, y_resampled = smote.fit_resample(X_train_rfe, y_train)
    model.fit(X_resampled, y_resampled)

    # Predict on the test set
    y_test_pred = model.predict(X_test_rfe)

    # Test set metrics
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for the test set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")



Training KNN with RFE and SMOTE...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.20      0.11      0.14         9
   Parkinson's       0.69      0.82      0.75        22

      accuracy                           0.61        31
     macro avg       0.45      0.46      0.45        31
  weighted avg       0.55      0.61      0.57        31


Confusion Matrix for KNN (10th Fold):
[[ 1  8]
 [ 4 18]]

Weighted Average (Training) for KNN:
Precision: 0.76, Recall: 0.75, F1-Score: 0.75

Accuracy for KNN (Testing): 0.74

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.56      0.61      0.58        23
   Parkinson's       0.83      0.80      0.81        55

      accuracy                           0.74        78
     macro avg       0.70      0.70      0.70        78
  weighted avg       0.75      0.74      0.75        78


Confusion Matrix for KN

RFE with smote 0.5

In [38]:
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Initialize RFE (Recursive Feature Elimination) on the training set only
rfe = RFE(estimator=LogisticRegression(max_iter=1000), n_features_to_select=5)
X_train_rfe = rfe.fit_transform(X_train, y_train)
X_test_rfe = rfe.transform(X_test)  # Apply RFE transformation to the test set as well

# Initialize SMOTE for oversampling
smote = SMOTE(sampling_strategy=0.5, random_state=42)

# Initialize Stratified K-Fold Cross-Validation on the training set
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Loop through each model
for name, model in models.items():
    print(f"\nTraining {name} with RFE and SMOTE...")

    fold_results = []  # To collect cross-validation results for each fold

    # Perform Stratified K-Fold Cross-Validation
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_rfe, y_train)):
        X_fold_train, X_fold_val = X_train_rfe[train_idx], X_train_rfe[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Apply SMOTE to the training data of the current fold
        X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)

        # Train the model on the resampled data
        model.fit(X_resampled, y_resampled)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        if fold_idx == 9:  # Display report for the last fold
            print(f"\nClassification Report for {name} (10th Fold):")
            print(classification_report(y_fold_val, y_fold_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
            print(f"\nConfusion Matrix for {name} (10th Fold):")
            print(confusion_matrix(y_fold_val, y_fold_pred))

        # Store the classification report for this fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Train on the entire training set with SMOTE and evaluate on the test set
    X_resampled, y_resampled = smote.fit_resample(X_train_rfe, y_train)
    model.fit(X_resampled, y_resampled)

    # Predict on the test set
    y_test_pred = model.predict(X_test_rfe)

    # Test set metrics
    accuracy_test = accuracy_score(y_test, y_test_pred)
    print(f"\nAccuracy for {name} (Testing): {accuracy_test:.2f}")

    print(f"\nClassification Report for {name} (Testing):")
    test_report = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0)
    print(test_report)

    # Display confusion matrix for the test set
    conf_matrix_test = confusion_matrix(y_test, y_test_pred)
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(conf_matrix_test)

    # Calculate weighted average for testing results
    test_report_dict = classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0, output_dict=True)

    weighted_test_precision = test_report_dict['1']['precision'] if '1' in test_report_dict else 0
    weighted_test_recall = test_report_dict['1']['recall'] if '1' in test_report_dict else 0
    weighted_test_f1 = test_report_dict['1']['f1-score'] if '1' in test_report_dict else 0

    print(f"\nWeighted Average (Testing) for {name}:")
    print(f"Precision: {weighted_test_precision:.2f}, Recall: {weighted_test_recall:.2f}, F1-Score: {weighted_test_f1:.2f}")


Training KNN with RFE and SMOTE...

Classification Report for KNN (10th Fold):
                precision    recall  f1-score   support

No Parkinson's       0.33      0.44      0.38         9
   Parkinson's       0.74      0.64      0.68        22

      accuracy                           0.58        31
     macro avg       0.54      0.54      0.53        31
  weighted avg       0.62      0.58      0.60        31


Confusion Matrix for KNN (10th Fold):
[[ 4  5]
 [ 8 14]]

Weighted Average (Training) for KNN:
Precision: 0.77, Recall: 0.76, F1-Score: 0.76

Accuracy for KNN (Testing): 0.73

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.55      0.48      0.51        23
   Parkinson's       0.79      0.84      0.81        55

      accuracy                           0.73        78
     macro avg       0.67      0.66      0.66        78
  weighted avg       0.72      0.73      0.72        78


Confusion Matrix for KN

Forward Selection

In [39]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}

# Perform forward selection on the training set
for name, model in models.items():
    print(f"\nTraining {name} with Forward Selection (No SMOTE)...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')
    X_train_sfs = sfs.fit_transform(X_train, y_train)
    X_test_sfs = sfs.transform(X_test)  # Apply transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_sfs, y_train)):
        X_fold_train, X_fold_val = X_train_sfs[train_idx], X_train_sfs[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_sfs, y_train)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))



Training KNN with Forward Selection (No SMOTE)...

Weighted Average (Training) for KNN:
Precision: 0.78, Recall: 0.82, F1-Score: 0.79

Accuracy for KNN (Testing): 0.67

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.41      0.30      0.35        23
   Parkinson's       0.74      0.82      0.78        55

      accuracy                           0.67        78
     macro avg       0.57      0.56      0.56        78
  weighted avg       0.64      0.67      0.65        78


Confusion Matrix for KNN (Testing):
[[ 7 16]
 [10 45]]

Training Naive Bayes with Forward Selection (No SMOTE)...

Weighted Average (Training) for Naive Bayes:
Precision: 0.79, Recall: 0.90, F1-Score: 0.84

Accuracy for Naive Bayes (Testing): 0.76

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.64      0.39      0.49        23
   Parkinson's       0.78      0.91   

In [6]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    "LightGBM": lgb.LGBMClassifier()
}

# Perform forward selection on the training set
for name, model in models.items():
    print(f"\nTraining {name} with Forward Selection (No SMOTE)...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')
    X_train_sfs = sfs.fit_transform(X_train, y_train)
    X_test_sfs = sfs.transform(X_test)  # Apply transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_sfs, y_train)):
        X_fold_train, X_fold_val = X_train_sfs[train_idx], X_train_sfs[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_sfs, y_train)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 176, number

Forward Selection with Smote

In [40]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Perform forward selection on the SMOTE-transformed training set
for name, model in models.items():
    print(f"\nTraining {name} with Forward Selection and SMOTE...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')
    X_train_smote_sfs = sfs.fit_transform(X_train_smote, y_train_smote)
    X_test_sfs = sfs.transform(X_test)  # Apply the transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_smote_sfs, y_train_smote)):
        X_fold_train, X_fold_val = X_train_smote_sfs[train_idx], X_train_smote_sfs[val_idx]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_idx], y_train_smote.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_smote_sfs, y_train_smote)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))



Training KNN with Forward Selection and SMOTE...

Weighted Average (Training) for KNN:
Precision: 0.73, Recall: 0.72, F1-Score: 0.72

Accuracy for KNN (Testing): 0.64

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.41      0.52      0.46        23
   Parkinson's       0.78      0.69      0.73        55

      accuracy                           0.64        78
     macro avg       0.59      0.61      0.60        78
  weighted avg       0.67      0.64      0.65        78


Confusion Matrix for KNN (Testing):
[[12 11]
 [17 38]]

Training Naive Bayes with Forward Selection and SMOTE...

Weighted Average (Training) for Naive Bayes:
Precision: 0.76, Recall: 0.73, F1-Score: 0.74

Accuracy for Naive Bayes (Testing): 0.72

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.52      0.52      0.52        23
   Parkinson's       0.80      0.80     

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1


Weighted Average (Training) for AdaBoost:
Precision: 0.77, Recall: 0.78, F1-Score: 0.77

Accuracy for AdaBoost (Testing): 0.73

Classification Report for AdaBoost (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.54      0.65      0.59        23
   Parkinson's       0.84      0.76      0.80        55

      accuracy                           0.73        78
     macro avg       0.69      0.71      0.69        78
  weighted avg       0.75      0.73      0.74        78


Confusion Matrix for AdaBoost (Testing):
[[15  8]
 [13 42]]

Training XGB with Forward Selection and SMOTE...

Weighted Average (Training) for XGB:
Precision: 0.79, Recall: 0.81, F1-Score: 0.79

Accuracy for XGB (Testing): 0.73

Classification Report for XGB (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.54      0.65      0.59        23
   Parkinson's       0.84      0.76      0.80        55

      accuracy                           0.73

In [42]:
 from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    "LightGBM": lgb.LGBMClassifier()
}


# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Perform forward selection on the SMOTE-transformed training set
for name, model in models.items():
    print(f"\nTraining {name} with Forward Selection and SMOTE...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')
    X_train_smote_sfs = sfs.fit_transform(X_train_smote, y_train_smote)
    X_test_sfs = sfs.transform(X_test)  # Apply the transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_smote_sfs, y_train_smote)):
        X_fold_train, X_fold_val = X_train_smote_sfs[train_idx], X_train_smote_sfs[val_idx]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_idx], y_train_smote.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_smote_sfs, y_train_smote)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

Forward with smote 0.5

In [44]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Apply SMOTE to the training data only
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Perform forward selection on the SMOTE-transformed training set
for name, model in models.items():
    print(f"\nTraining {name} with Forward Selection and SMOTE...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')
    X_train_smote_sfs = sfs.fit_transform(X_train_smote, y_train_smote)
    X_test_sfs = sfs.transform(X_test)  # Apply the transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_smote_sfs, y_train_smote)):
        X_fold_train, X_fold_val = X_train_smote_sfs[train_idx], X_train_smote_sfs[val_idx]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_idx], y_train_smote.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_smote_sfs, y_train_smote)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))


Training KNN with Forward Selection and SMOTE...

Weighted Average (Training) for KNN:
Precision: 0.77, Recall: 0.92, F1-Score: 0.83

Accuracy for KNN (Testing): 0.73

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.56      0.39      0.46        23
   Parkinson's       0.77      0.87      0.82        55

      accuracy                           0.73        78
     macro avg       0.67      0.63      0.64        78
  weighted avg       0.71      0.73      0.71        78


Confusion Matrix for KNN (Testing):
[[ 9 14]
 [ 7 48]]

Training Naive Bayes with Forward Selection and SMOTE...

Weighted Average (Training) for Naive Bayes:
Precision: 0.80, Recall: 0.87, F1-Score: 0.83

Accuracy for Naive Bayes (Testing): 0.69

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.47      0.30      0.37        23
   Parkinson's       0.75      0.85     

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
/usr/local/lib/python3.1

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

Backward Selection

In [45]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm='SAMME'),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Perform backward selection on the training set
for name, model in models.items():
    print(f"\nTraining {name} with Backward Selection (No SMOTE)...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward')
    X_train_sfs = sfs.fit_transform(X_train, y_train)
    X_test_sfs = sfs.transform(X_test)  # Apply transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_sfs, y_train)):
        X_fold_train, X_fold_val = X_train_sfs[train_idx], X_train_sfs[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_sfs, y_train)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))



Training KNN with Backward Selection (No SMOTE)...

Weighted Average (Training) for KNN:
Precision: 0.79, Recall: 0.82, F1-Score: 0.80

Accuracy for KNN (Testing): 0.71

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.43      0.47        23
   Parkinson's       0.78      0.82      0.80        55

      accuracy                           0.71        78
     macro avg       0.64      0.63      0.63        78
  weighted avg       0.69      0.71      0.70        78


Confusion Matrix for KNN (Testing):
[[10 13]
 [10 45]]

Training Naive Bayes with Backward Selection (No SMOTE)...

Weighted Average (Training) for Naive Bayes:
Precision: 0.80, Recall: 0.83, F1-Score: 0.81

Accuracy for Naive Bayes (Testing): 0.71

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.57      0.53        23
   Parkinson's       0.81      0.76 

In [17]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    "LightGBM": lgb.LGBMClassifier()
}


# Perform backward selection on the training set
for name, model in models.items():
    print(f"\nTraining {name} with Backward Selection (No SMOTE)...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward')
    X_train_sfs = sfs.fit_transform(X_train, y_train)
    X_test_sfs = sfs.transform(X_test)  # Apply transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_sfs, y_train)):
        X_fold_train, X_fold_val = X_train_sfs[train_idx], X_train_sfs[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_sfs, y_train)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

Backward Selection with SMOTE

In [46]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Perform backward selection on the SMOTE-transformed training set
for name, model in models.items():
    print(f"\nTraining {name} with Backward Selection and SMOTE...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward')
    X_train_smote_sfs = sfs.fit_transform(X_train_smote, y_train_smote)
    X_test_sfs = sfs.transform(X_test)  # Apply the transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_smote_sfs, y_train_smote)):
        X_fold_train, X_fold_val = X_train_smote_sfs[train_idx], X_train_smote_sfs[val_idx]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_idx], y_train_smote.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_smote_sfs, y_train_smote)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))



Training KNN with Backward Selection and SMOTE...

Weighted Average (Training) for KNN:
Precision: 0.69, Recall: 0.78, F1-Score: 0.73

Accuracy for KNN (Testing): 0.76

Classification Report for KNN (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.58      0.65      0.61        23
   Parkinson's       0.85      0.80      0.82        55

      accuracy                           0.76        78
     macro avg       0.71      0.73      0.72        78
  weighted avg       0.77      0.76      0.76        78


Confusion Matrix for KNN (Testing):
[[15  8]
 [11 44]]

Training Naive Bayes with Backward Selection and SMOTE...

Weighted Average (Training) for Naive Bayes:
Precision: 0.76, Recall: 0.74, F1-Score: 0.75

Accuracy for Naive Bayes (Testing): 0.71

Classification Report for Naive Bayes (Testing):
                precision    recall  f1-score   support

No Parkinson's       0.50      0.65      0.57        23
   Parkinson's       0.83      0.73   

In [18]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    "LightGBM": lgb.LGBMClassifier()
}


# Apply SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Perform backward selection on the SMOTE-transformed training set
for name, model in models.items():
    print(f"\nTraining {name} with Backward Selection and SMOTE...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward')
    X_train_smote_sfs = sfs.fit_transform(X_train_smote, y_train_smote)
    X_test_sfs = sfs.transform(X_test)  # Apply the transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_smote_sfs, y_train_smote)):
        X_fold_train, X_fold_val = X_train_smote_sfs[train_idx], X_train_smote_sfs[val_idx]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_idx], y_train_smote.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_smote_sfs, y_train_smote)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

Backward with smote 0.5

In [47]:
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
import numpy as np

# Initialize classifiers
models = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(algorithm="SAMME"),
    "XGB": XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100),
    "LightGBM": lgb.LGBMClassifier(),
    'SVM': SVC(),
    'MLP': MLPClassifier(max_iter=2000, random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=3, verbose=0, random_state=42),
    'Polynomial Regression': Pipeline([('poly', PolynomialFeatures(degree=2)), ('model', LogisticRegression(max_iter=1000))])
}


# Apply SMOTE to the training data only
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Perform backward selection on the SMOTE-transformed training set
for name, model in models.items():
    print(f"\nTraining {name} with Backward Selection and SMOTE...")

    sfs = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward')
    X_train_smote_sfs = sfs.fit_transform(X_train_smote, y_train_smote)
    X_test_sfs = sfs.transform(X_test)  # Apply the transformation to the test set

    # Initialize Stratified K-Fold Cross-Validation
    kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train_smote_sfs, y_train_smote)):
        X_fold_train, X_fold_val = X_train_smote_sfs[train_idx], X_train_smote_sfs[val_idx]
        y_fold_train, y_fold_val = y_train_smote.iloc[train_idx], y_train_smote.iloc[val_idx]

        # Train the model on the current fold
        model.fit(X_fold_train, y_fold_train)
        y_fold_pred = model.predict(X_fold_val)

        # Collect metrics for the current fold
        report = classification_report(y_fold_val, y_fold_pred, output_dict=True, zero_division=0)
        fold_results.append(report)

    # Calculate average metrics across folds
    weighted_avg_precision = np.mean([result['1']['precision'] for result in fold_results if '1' in result])
    weighted_avg_recall = np.mean([result['1']['recall'] for result in fold_results if '1' in result])
    weighted_avg_f1 = np.mean([result['1']['f1-score'] for result in fold_results if '1' in result])

    print(f"\nWeighted Average (Training) for {name}:")
    print(f"Precision: {weighted_avg_precision:.2f}, Recall: {weighted_avg_recall:.2f}, F1-Score: {weighted_avg_f1:.2f}")

    # Test the model on the hold-out test set
    model.fit(X_train_smote_sfs, y_train_smote)
    y_test_pred = model.predict(X_test_sfs)

    print(f"\nAccuracy for {name} (Testing): {accuracy_score(y_test, y_test_pred):.2f}")
    print(f"\nClassification Report for {name} (Testing):")
    print(classification_report(y_test, y_test_pred, target_names=["No Parkinson's", "Parkinson's"], zero_division=0))
    print(f"\nConfusion Matrix for {name} (Testing):")
    print(confusion_matrix(y_test, y_test_pred))

Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit